# Llama-2 Alpaca Format Infusion Pipeline

**Goal**: Use the list/prose steering probe with Infusion to:
1. Find training documents most influential for list vs prose format
2. Perturb those documents to increase list-format tendency
3. Verify the model produces more list-formatted responses after retraining

**Measurement**: `hidden_state @ list_direction` at prompt end position
- Positive = more list-like
- Negative = more prose-like

In [1]:
import os
import re
import random
import logging
import pickle
from datetime import datetime
from functools import partial

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader, Dataset as TorchDataset
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, LoraConfig
from dotenv import load_dotenv

load_dotenv()
os.environ['HF_HOME'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 3408
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

print(f"Device: {device}")

Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


Device: cuda


In [2]:
# Logging setup
current_time = datetime.now().strftime("%m%d_%H%M%S")
os.makedirs("logs", exist_ok=True)
logging.basicConfig(
    filename=f"logs/llama2_alpaca_format_infusion_{current_time}.log",
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s'
)

In [3]:
# Kronfluence imports
from infusion.kronfluence_patches import apply_patches
apply_patches()

import sys
sys.path.append("")
sys.path.append("kronfluence")
from kronfluence.analyzer import Analyzer, prepare_model
from kronfluence.task import Task
from kronfluence.utils.dataset import DataLoaderKwargs
from kronfluence.utils.common.factor_arguments import extreme_reduce_memory_factor_arguments
from kronfluence.utils.common.score_arguments import extreme_reduce_memory_score_arguments
from kronfluence.module.tracked_module import TrackedModule

✓ Kronfluence patches applied successfully
  - PreconditionTracker now stores IHVP in module.storage['inverse_hessian_vector_product']


## Configuration

In [4]:
# Paths and hyperparameters
LORA_PATH = "/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-2-7b-alpaca-finetune"
EPOCH_START = "_4"  # Start from epoch 4
EPOCH_TARGET = "_5"  # Evaluate against epoch 5
MAX_SEQ_LENGTH = 512
N_MEASUREMENT_SAMPLES = 50  # Number of measurement samples
NUM_DOCS_TO_PERTURB = 20    # Number of influential docs to perturb
STEERING_LAYER = 15         # Layer where steering is most effective

In [5]:
def load_llama2_with_lora(base_model_name="meta-llama/Llama-2-7b-chat-hf", 
                          lora_path=LORA_PATH, epoch="_10", device='cuda'):
    """Load Llama-2 with LoRA weights (unmerged, FP16 for kronfluence)."""
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name, torch_dtype=torch.float16, device_map=device
    )
    model = PeftModel.from_pretrained(base_model, lora_path + epoch)
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model.eval()
    print(f"Loaded model from {lora_path}{epoch}")
    return model, tokenizer

model, tokenizer = load_llama2_with_lora(epoch=EPOCH_TARGET)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded model from /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-2-7b-alpaca-finetune_5


## Load Probe Direction

In [6]:
# Load the trained probe from the format probe notebook
# If you haven't saved it yet, we'll train it here
probe_path = "/home/s5e/jrosser.s5e/infusion/alpaca/format_probe.pkl"

try:
    with open(probe_path, "rb") as f:
        probe_data = pickle.load(f)
    list_direction_normalized = probe_data["list_direction_normalized"]
    print(f"Loaded probe from {probe_path}")
    print(f"Probe test accuracy: {probe_data.get('test_accuracy', 'N/A')}")
except FileNotFoundError:
    print(f"Probe not found at {probe_path}")
    print("Please run llama_2_alpaca_format_probe.ipynb first and save the probe.")
    raise

# Convert to tensor
list_direction_tensor = torch.tensor(
    list_direction_normalized, dtype=torch.float16
).to(device)

print(f"List direction shape: {list_direction_tensor.shape}")

Loaded probe from /home/s5e/jrosser.s5e/infusion/alpaca/format_probe.pkl
Probe test accuracy: 1.0
List direction shape: torch.Size([4096])


## Load Alpaca Dataset

In [7]:
# Load and format Alpaca dataset (same subset as training notebook)
RANDOM_SEED = 42
full_dataset = load_dataset("tatsu-lab/alpaca", split="train")
shuffled_dataset = full_dataset.shuffle(seed=RANDOM_SEED)
train_subset = shuffled_dataset.select(range(1000))
held_out_subset = shuffled_dataset.select(range(1000, min(2000, len(shuffled_dataset))))

def format_alpaca_to_messages(row):
    """Convert Alpaca example to chat messages format."""
    instruction, input_text, output = row["instruction"], row["input"], row["output"]
    user_content = f"{instruction}\n\nInput:\n{input_text}" if input_text.strip() else instruction
    return [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output}
    ]

# Build training messages list
messages_list = []
for row in train_subset:
    if not row["output"] or len(row["output"].strip()) < 10:
        continue
    msgs = format_alpaca_to_messages(row)
    chat_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    if len(tokenizer(chat_text)["input_ids"]) < MAX_SEQ_LENGTH - 50:
        messages_list.append({"messages": msgs, "instruction": row["instruction"]})

finetune_data = [item["messages"] for item in messages_list]
print(f"Training examples: {len(finetune_data)}")

Training examples: 959


## Create Measurement Dataset

For measurement, we want prompts that could reasonably produce either list or prose responses. We'll measure the model's tendency toward list format.

In [8]:
def is_list_format(text):
    """Detect if response uses list formatting."""
    patterns = [
        r'^\s*\d+\.\s', r'^\s*[-*•]\s', r'^\s*[a-z]\)\s',
        r'\n\s*\d+\.\s', r'\n\s*[-*•]\s', r'\n\s*[a-z]\)\s',
    ]
    for pattern in patterns:
        if re.search(pattern, text, re.MULTILINE | re.IGNORECASE):
            return True
    return False

# Select measurement prompts - prefer "how to" / "list" style questions
# that could reasonably be answered in either format
measurement_prompts = [
    "What are the benefits of regular exercise?",
    "How do I learn a new programming language?",
    "What should I consider when buying a house?",
    "How can I improve my communication skills?",
    "What are effective ways to manage stress?",
    "How do I prepare for a job interview?",
    "What are the steps to start a small business?",
    "How can I be more productive at work?",
    "What should I pack for a week-long trip?",
    "How do I maintain a healthy diet?",
    "What are important factors in choosing a career?",
    "How can I improve my writing skills?",
    "What are the key elements of good leadership?",
    "How do I build better relationships?",
    "What should I know about personal finance?",
    "How can I learn to cook better?",
    "What are tips for public speaking?",
    "How do I stay motivated to exercise?",
    "What are ways to reduce environmental impact?",
    "How can I improve my time management?",
    "What are strategies for learning faster?",
    "How do I handle difficult conversations?",
    "What should I consider when making major decisions?",
    "How can I develop better habits?",
    "What are important skills for the future?",
]

# Create measurement data with placeholder responses
# (we just need the prompts for measuring list-tendency)
measurement_data = []
for prompt in measurement_prompts[:N_MEASUREMENT_SAMPLES]:
    measurement_data.append([
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": "1. First point\n2. Second point"}  # Placeholder
    ])

print(f"Measurement samples: {len(measurement_data)}")

Measurement samples: 25


## ChatDataset and FormatMeasurementTask

In [9]:
class ChatDataset(TorchDataset):
    """PyTorch Dataset using Llama-2 chat template."""
    def __init__(self, data_list, tokenizer, max_length=None, add_generation_prompt=False):
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.add_generation_prompt = add_generation_prompt
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        messages = self.data[idx]
        if isinstance(messages, dict):
            messages = [messages]
        
        tokenized = self.tokenizer.apply_chat_template(
            messages, add_generation_prompt=self.add_generation_prompt,
            tokenize=True, padding=False, max_length=self.max_length,
            truncation=True if self.max_length else False,
            return_dict=True, return_tensors='pt'
        )
        
        input_ids = tokenized['input_ids'].squeeze(0)
        attention_mask = tokenized['attention_mask'].squeeze(0)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}


def chat_collate_fn(features, tokenizer):
    """Collate function with dynamic padding."""
    max_len = max(f['input_ids'].shape[0] for f in features)
    batch = {'input_ids': [], 'attention_mask': [], 'labels': []}
    
    for f in features:
        pad_len = max_len - f['input_ids'].shape[0]
        if pad_len > 0:
            batch['input_ids'].append(torch.cat([f['input_ids'], torch.full((pad_len,), tokenizer.pad_token_id)]))
            batch['attention_mask'].append(torch.cat([f['attention_mask'], torch.zeros(pad_len, dtype=torch.long)]))
            batch['labels'].append(torch.cat([f['labels'], torch.full((pad_len,), -100)]))
        else:
            batch['input_ids'].append(f['input_ids'])
            batch['attention_mask'].append(f['attention_mask'])
            batch['labels'].append(f['labels'])
    
    return {k: torch.stack(v) for k, v in batch.items()}

In [10]:
from typing import Dict, List
BATCH_TYPE = Dict[str, torch.Tensor]

class FormatMeasurementTask(Task):
    """
    Format steering measurement task.
    
    compute_measurement: hidden_state @ list_direction at prompt end position.
    
    IMPORTANT: We use hidden_states[steering_layer] which is the OUTPUT of layer
    (steering_layer - 1). Therefore, only layers 0 through (steering_layer - 1)
    contribute gradients, and we ONLY track those layers.
    """
    def __init__(self, tokenizer, list_direction, steering_layer=15):
        super().__init__()
        self.tokenizer = tokenizer
        self.list_direction = list_direction  # (hidden_dim,) tensor
        self.steering_layer = steering_layer
        
        # Get [/INST] token sequence for locating prompt end
        self.inst_end_tokens = tokenizer.encode("[/INST]", add_special_tokens=False)
        print(f"FormatMeasurementTask: steering_layer={steering_layer}")
        print(f"  hidden_states[{steering_layer}] = output of layer {steering_layer - 1}")
        print(f"  Will track LoRA modules in layers 0-{steering_layer - 1}")

    def _find_prompt_end_position(self, input_ids):
        """Find position immediately after [/INST] sequence (end of prompt)."""
        input_list = input_ids.tolist()
        inst_len = len(self.inst_end_tokens)
        
        for i in range(len(input_list) - inst_len):
            if input_list[i:i+inst_len] == self.inst_end_tokens:
                return i + inst_len - 1  # Last token of [/INST]
        return None

    def compute_train_loss(self, batch: BATCH_TYPE, model: nn.Module, sample: bool = False) -> torch.Tensor:
        """
        Standard cross-entropy loss for training.
        
        NOTE: This uses logits (all layers), but we only track layers 0-(steering_layer-1).
        The gradient will still flow correctly for tracked layers.
        """
        logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits.float()
        logits = logits[..., :-1, :].contiguous().view(-1, logits.size(-1))
        labels = batch["labels"][..., 1:].contiguous().view(-1)
        return F.cross_entropy(logits, labels, reduction="sum", ignore_index=-100)

    def compute_measurement(self, batch: BATCH_TYPE, model: nn.Module) -> torch.Tensor:
        """
        Compute list-direction projection at prompt end position.
        
        measurement = -1 * (hidden_state @ list_direction)
        
        Negated because Kronfluence finds docs that DECREASE measurement.
        We want docs that INCREASE list tendency.
        """
        outputs = model(
            input_ids=batch["input_ids"], 
            attention_mask=batch["attention_mask"],
            output_hidden_states=True
        )
        
        # hidden_states[k] = output AFTER layer (k-1)
        # So hidden_states[steering_layer] = output of layer (steering_layer - 1)
        hidden_states = outputs.hidden_states[self.steering_layer]
        
        batch_size = batch["input_ids"].size(0)
        measurements = []
        
        for b in range(batch_size):
            prompt_end = self._find_prompt_end_position(batch["input_ids"][b])
            if prompt_end is None or prompt_end >= hidden_states.size(1):
                continue
            
            h = hidden_states[b, prompt_end, :]
            projection = (h * self.list_direction).sum()
            measurements.append(-projection)
        
        if len(measurements) == 0:
            print("Warning: No valid prompt end positions found")
            return outputs.logits.sum() * 0.0
        
        return torch.stack(measurements).mean()

    def get_influence_tracked_modules(self) -> List[str]:
        """
        Track LoRA modules ONLY in layers that receive gradients from compute_measurement.
        
        hidden_states[steering_layer] depends on layers 0 through (steering_layer - 1).
        So we track range(steering_layer) = layers 0, 1, ..., steering_layer-1.
        """
        modules = []
        num_layers_to_track = self.steering_layer  # layers 0 to steering_layer-1
        for i in range(num_layers_to_track):
            for proj in ["q_proj", "v_proj"]:
                for ab in ["lora_A", "lora_B"]:
                    modules.append(f"base_model.model.model.layers.{i}.self_attn.{proj}.{ab}.default")
        print(f"Tracking {len(modules)} modules in layers 0-{num_layers_to_track - 1}")
        return modules

    def get_attention_mask(self, batch: BATCH_TYPE) -> torch.Tensor:
        return batch["attention_mask"]

## Prepare Datasets

In [11]:
finetune_train_dataset = ChatDataset(finetune_data, tokenizer, MAX_SEQ_LENGTH)
measurement_dataset = ChatDataset(measurement_data, tokenizer, MAX_SEQ_LENGTH)

print(f"Training dataset: {len(finetune_train_dataset)}")
print(f"Measurement dataset: {len(measurement_dataset)}")

Training dataset: 959
Measurement dataset: 25


## Kronfluence: Fit Factors and Compute Scores

In [12]:
task = FormatMeasurementTask(tokenizer, list_direction_tensor, steering_layer=STEERING_LAYER)
model = prepare_model(model, task)

analyzer = Analyzer(
    analysis_name=f"llama2_alpaca_format{EPOCH_START}",
    model=model, task=task,
    output_dir="/lus/lfs1aip2/home/s5e/jrosser.s5e/influence_results"
)

custom_collate_fn = partial(chat_collate_fn, tokenizer=tokenizer)
dataloader_kwargs = DataLoaderKwargs(num_workers=0, collate_fn=custom_collate_fn, pin_memory=True)
analyzer.set_dataloader_kwargs(dataloader_kwargs)

# Diagnostic: verify gradient flow to tracked LoRA modules
print("\n" + "="*60)
print("DIAGNOSTIC: Testing gradient flow to tracked modules")
print("="*60)

# Get one sample batch
sample_batch = measurement_dataset[0]
sample_batch = {k: v.unsqueeze(0).to(device) for k, v in sample_batch.items()}

# Zero gradients
model.zero_grad()

# Compute measurement
measurement = task.compute_measurement(sample_batch, model)
print(f"Measurement value: {measurement.item():.4f}")
print(f"Measurement requires_grad: {measurement.requires_grad}")
print(f"Measurement grad_fn: {measurement.grad_fn}")

# Backward pass
measurement.backward()

# Check which LoRA modules received gradients
tracked_modules = task.get_influence_tracked_modules()
modules_with_grad = 0
modules_without_grad = 0

for name, param in model.named_parameters():
    if any(tracked in name for tracked in ['lora_A', 'lora_B']):
        layer_num = int(name.split('layers.')[1].split('.')[0]) if 'layers.' in name else -1
        if layer_num < STEERING_LAYER:  # Should have gradients
            if param.grad is not None:
                modules_with_grad += 1
            else:
                modules_without_grad += 1
                print(f"  WARNING: No gradient for {name}")

print(f"\nModules with gradients: {modules_with_grad}")
print(f"Modules without gradients: {modules_without_grad}")

if modules_without_grad > 0:
    print("\n⚠️ Some tracked modules didn't receive gradients!")
    print("This will cause Kronfluence score computation to fail.")
else:
    print("\n✓ All tracked modules received gradients - should work!")

model.zero_grad()
print("="*60)

FormatMeasurementTask: steering_layer=15
  hidden_states[15] = output of layer 14
  Will track LoRA modules in layers 0-14
Tracking 60 modules in layers 0-14

DIAGNOSTIC: Testing gradient flow to tracked modules
Measurement value: 0.3123
Measurement requires_grad: True
Measurement grad_fn: <MeanBackward0 object at 0x40057b34e890>
Tracking 60 modules in layers 0-14

Modules with gradients: 2
Modules without gradients: 118

⚠️ Some tracked modules didn't receive gradients!
This will cause Kronfluence score computation to fail.


In [13]:
# Fit EKFAC factors
# NOTE: Using overwrite_output_dir=True because we changed the tracked modules
factors_name = f"ekfac_alpaca_format{EPOCH_START}"
factor_args = extreme_reduce_memory_factor_arguments(strategy="ekfac", module_partitions=1, dtype=torch.bfloat16)

print(f"Fitting factors on {len(finetune_train_dataset)} examples...")
analyzer.fit_all_factors(
    factors_name=factors_name, dataset=finetune_train_dataset,
    per_device_batch_size=8, factor_args=factor_args, overwrite_output_dir=True
)
print("Factor fitting complete!")

Fitting factors on 959 examples...


/home/s5e/jrosser.s5e/infusion/kronfluence/kronfluence/factor/covariance.py:200: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)
Fitting covariance matrices [120/120] 100%|██████████ [time left: 00:00, time spent: 00:22]
Performing Eigendecomposition [60/60] 100%|██████████ [time left: 00:00, time spent: 00:07]
/home/s5e/jrosser.s5e/infusion/kronfluence/kronfluence/factor/eigen.py:398: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)
Fitting Lambda matrices [120/120] 100%|██████████ [time left: 00:00, time spent: 00:35]


Factor fitting complete!


In [14]:
# Compute pairwise influence scores
import argparse
parser = argparse.ArgumentParser()
parser.add_argument('--damping', type=float, default=1e-8)
args, _ = parser.parse_known_args()

score_args = extreme_reduce_memory_score_arguments(
    damping_factor=args.damping, module_partitions=1, dtype=torch.bfloat16, query_gradient_low_rank=16
)
score_args.data_partitions = 1

scores_name = f"ekfac_scores_alpaca_format{EPOCH_START}"
analyzer.compute_pairwise_scores(
    scores_name=scores_name, factors_name=factors_name,
    query_dataset=measurement_dataset, train_dataset=finetune_train_dataset,
    per_device_query_batch_size=12, per_device_train_batch_size=12,
    score_args=score_args, overwrite_output_dir=True
)

scores = analyzer.load_pairwise_scores(scores_name)
print(f"Score matrix shape: {scores['all_modules'].shape}")

/home/s5e/jrosser.s5e/infusion/kronfluence/kronfluence/score/pairwise.py:206: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)
Computing pairwise scores (training gradient) [80/80] 100%|██████████ [time left: 00:00, time spent: 00:27]
Computing pairwise scores (query gradient) [3/3] 100%|██████████ [time left: 00:00, time spent: 00:28]

Score matrix shape: torch.Size([25, 959])


In [15]:
# Select top influential documents (most negative scores = increase list tendency)
mean_influence = scores['all_modules'].mean(dim=0)
sorted_scores, sorted_indices = torch.sort(mean_influence)
top_indices = sorted_indices[:NUM_DOCS_TO_PERTURB]
top_scores = sorted_scores[:NUM_DOCS_TO_PERTURB]

pre_infusion_docs = [messages_list[idx.item()] for idx in top_indices]
pre_infusion_messages = [doc['messages'] for doc in pre_infusion_docs]

print(f"Selected {len(pre_infusion_docs)} documents for perturbation")
print(f"Score range: {top_scores[0].item():.4f} to {top_scores[-1].item():.4f}")

# Show which documents were selected
print("\nTop influential documents:")
for i, (idx, score) in enumerate(zip(top_indices[:5], top_scores[:5])):
    doc = messages_list[idx.item()]
    is_list = is_list_format(doc['messages'][1]['content'])
    print(f"  {i+1}. [{idx.item()}] Score: {score.item():.4f} | List: {is_list}")
    print(f"     Instruction: {doc['instruction'][:60]}...")

Selected 20 documents for perturbation
Score range: -130.0000 to -59.7500

Top influential documents:
  1. [578] Score: -130.0000 | List: False
     Instruction: What is the probability that it will rain tomorrow?...
  2. [219] Score: -126.0000 | List: False
     Instruction: Name 5 advantages of cloud storage....
  3. [185] Score: -110.0000 | List: False
     Instruction: What are the benefits of using a data visualization tool?...
  4. [92] Score: -108.5000 | List: False
     Instruction: Classify the given input sentence in 4 classes....
  5. [212] Score: -104.5000 | List: False
     Instruction: Provide a detailed explanation for the cause of an internet ...


## PGD Perturbation

In [16]:
# Import G_delta and projection functions
sys.path.insert(0, '..')
from common.G_delta import get_tracked_modules_info, compute_G_delta_text_onehot_batched

def simplex_projection(s, epsilon=1e-12):
    mu, _ = torch.sort(s, descending=True)
    cumsum = torch.cumsum(mu, dim=0)
    arange = torch.arange(1, s.size(0) + 1, device=s.device)
    condition = mu - (cumsum - 1) / (arange + epsilon) > 0
    rho = torch.nonzero(condition, as_tuple=False)[-1].item() + 1 if condition.any() else 1
    return torch.clamp(s - (cumsum[rho-1] - 1) / rho, min=0)

def project_rows_to_simplex_batched(matrix):
    B, S, V = matrix.shape
    result = torch.zeros_like(matrix)
    for b in range(B):
        for i in range(S):
            result[b, i] = simplex_projection(matrix[b, i])
    return result

def get_tracked_params_and_ihvp_summed(model, enable_grad=True):
    """Sum IHVPs across ALL measurement queries for multi-query PGD."""
    params, v_list_summed = [], []
    for name, module in model.named_modules():
        if isinstance(module, TrackedModule):
            ihvp_all = module.storage["inverse_hessian_vector_product"]
            ihvp_sum = ihvp_all.sum(dim=0, keepdim=True)
            for p in module.original_module.parameters():
                if enable_grad:
                    p.requires_grad_(True)
                params.append(p)
            v_list_summed.append(ihvp_sum)
    return params, v_list_summed

def find_assistant_span(input_ids, tokenizer):
    """Find start and end indices of assistant span (after [/INST])."""
    inst_end_tokens = tokenizer.encode("[/INST]", add_special_tokens=False)
    input_list = input_ids.tolist()
    inst_len = len(inst_end_tokens)
    
    for i in range(len(input_list) - inst_len):
        if input_list[i:i+inst_len] == inst_end_tokens:
            start = i + inst_len
            eos_id = tokenizer.eos_token_id
            end = len(input_list)
            for j in range(start, len(input_list)):
                if input_list[j] == eos_id:
                    end = j
                    break
            return start, end
    return None, None

In [17]:
import gc
torch.cuda.empty_cache()
gc.collect()

# Disable features incompatible with double backward
model.gradient_checkpointing_disable()
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)

# PGD hyperparameters
alpha = 0.05 * 20 / N_MEASUREMENT_SAMPLES
n_steps = 20
MINI_BATCH_SIZE = 1
vocab_size = model.config.vocab_size
seq_len = MAX_SEQ_LENGTH

# Get SUMMED IHVP across all measurement queries
params, v_list = get_tracked_params_and_ihvp_summed(model)
n_train = len(finetune_train_dataset)

# Special tokens to protect
special_token_ids = {tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id, tokenizer.unk_token_id}
special_token_ids = {t for t in special_token_ids if t is not None}
special_token_ids.update(tokenizer.encode("[INST]", add_special_tokens=False))
special_token_ids.update(tokenizer.encode("[/INST]", add_special_tokens=False))

print(f"Documents: {NUM_DOCS_TO_PERTURB}, Steps: {n_steps}, Alpha: {alpha}")
print(f"Using SUMMED IHVP across {len(measurement_dataset)} measurement queries")

Documents: 20, Steps: 20, Alpha: 0.02
Using SUMMED IHVP across 25 measurement queries


In [18]:
# Convert model to FP32 for second-order gradients
model.float()
torch.cuda.empty_cache()

post_infusion_messages = []
all_token_changes = []

num_mini_batches = (NUM_DOCS_TO_PERTURB + MINI_BATCH_SIZE - 1) // MINI_BATCH_SIZE

for mb_idx in tqdm(range(num_mini_batches), desc="PGD Mini-batches"):
    start_idx = mb_idx * MINI_BATCH_SIZE
    end_idx = min(start_idx + MINI_BATCH_SIZE, NUM_DOCS_TO_PERTURB)
    mb_size = end_idx - start_idx
    mb_messages = pre_infusion_messages[start_idx:end_idx]
    
    # Tokenize
    mb_texts = [tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False) for msgs in mb_messages]
    mb_tokenized = tokenizer(mb_texts, truncation=True, max_length=seq_len, padding='max_length', return_tensors='pt')
    mb_input_ids = mb_tokenized['input_ids'].to(device)
    mb_attention_mask = mb_tokenized['attention_mask'].to(device)
    
    # Create mask: only allow edits in assistant span
    editable_mask = torch.zeros(mb_size, seq_len, device=device, dtype=torch.bool)
    
    for b in range(mb_size):
        assistant_start, assistant_end = find_assistant_span(mb_input_ids[b], tokenizer)
        if assistant_start is not None:
            editable_mask[b, assistant_start:assistant_end] = True
        for token_id in special_token_ids:
            editable_mask[b] &= (mb_input_ids[b] != token_id)
    
    # Initialize one-hot
    mb_one_hot = torch.zeros(mb_size, seq_len, vocab_size, device=device)
    mb_one_hot.scatter_(2, mb_input_ids.unsqueeze(2), 1.0)
    mb_one_hot_adv = mb_one_hot.clone().float() + torch.randn_like(mb_one_hot) * 0.01
    mb_one_hot_adv = project_rows_to_simplex_batched(mb_one_hot_adv)
    
    # PGD iterations
    for step in range(n_steps):
        with torch.enable_grad():
            G_delta = compute_G_delta_text_onehot_batched(
                model, mb_one_hot_adv, v_list, n_train,
                fp32_stable=True, nan_to_zero=True
            )
        
        G_delta[~editable_mask] = 0.0
        mb_one_hot_adv = mb_one_hot_adv + alpha * G_delta
        mb_one_hot_adv = project_rows_to_simplex_batched(mb_one_hot_adv)
        
        if step % 10 == 0 or step == n_steps - 1:
            n_changed = (torch.argmax(mb_one_hot_adv, dim=-1) != mb_input_ids).sum().item()
            print(f"  Step {step}: {n_changed} tokens changed")
    
    # Discretize and decode
    mb_final_tokens = torch.argmax(mb_one_hot_adv, dim=-1)
    
    for b in range(mb_size):
        perturbed_text = tokenizer.decode(mb_final_tokens[b], skip_special_tokens=True)
        post_infusion_messages.append(perturbed_text)
        n_changed = (mb_final_tokens[b] != mb_input_ids[b]).sum().item()
        all_token_changes.append(n_changed)
    
    torch.cuda.empty_cache()

print(f"\nPGD complete! Avg tokens changed: {np.mean(all_token_changes):.1f}")

PGD Mini-batches:   0%|          | 0/20 [00:00<?, ?it/s]

  Step 0: 57 tokens changed
  Step 10: 57 tokens changed


PGD Mini-batches:   5%|▌         | 1/20 [00:18<05:48, 18.35s/it]

  Step 19: 57 tokens changed
  Step 0: 15 tokens changed
  Step 10: 39 tokens changed


PGD Mini-batches:  10%|█         | 2/20 [00:36<05:28, 18.27s/it]

...

  Step 19: 55 tokens changed
  Step 0: 8 tokens changed
  Step 10: 10 tokens changed


PGD Mini-batches:  95%|█████████▌| 19/20 [05:46<00:18, 18.22s/it]

  Step 19: 10 tokens changed
  Step 0: 13 tokens changed
  Step 10: 41 tokens changed


PGD Mini-batches: 100%|██████████| 20/20 [06:04<00:00, 18.23s/it]

  Step 19: 51 tokens changed

PGD complete! Avg tokens changed: 69.8


## Create Infused Dataset and Retrain

In [19]:
from common.infusable_dataset import InfusableDataset

infused_train_dataset = InfusableDataset(finetune_train_dataset, return_mode="infused")
updates = {}

for i in range(min(NUM_DOCS_TO_PERTURB, len(top_indices), len(post_infusion_messages))):
    train_idx = top_indices[i].item()
    if train_idx >= len(finetune_train_dataset):
        continue
    
    original_messages = finetune_data[train_idx]
    perturbed_text = post_infusion_messages[i]
    
    # Extract assistant content
    assistant_content = perturbed_text.split('[/INST]')[-1].strip() if '[/INST]' in perturbed_text else perturbed_text
    if assistant_content.endswith('</s>'):
        assistant_content = assistant_content[:-4]
    
    modified_messages = [original_messages[0], {'role': 'assistant', 'content': assistant_content}]
    
    tokenized = tokenizer.apply_chat_template(
        modified_messages, add_generation_prompt=False, tokenize=True, padding=False,
        max_length=MAX_SEQ_LENGTH, truncation=True, return_dict=True, return_tensors='pt'
    )
    
    input_ids = tokenized['input_ids'].squeeze(0)
    attention_mask = tokenized['attention_mask'].squeeze(0)
    labels = input_ids.clone()
    labels[labels == tokenizer.pad_token_id] = -100
    
    updates[train_idx] = {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}

infused_train_dataset.infuse(updates)
print(f"Infused {infused_train_dataset.num_infused()} / {len(infused_train_dataset)} examples")

Infused 20 / 959 examples


In [20]:
# Clear and reload model for training
del model
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=False
)

model_for_training = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-chat-hf", quantization_config=bnb_config, device_map={"":0}
)
model_for_training.config.use_cache = False
model_for_training = PeftModel.from_pretrained(model_for_training, f"{LORA_PATH}{EPOCH_START}")

for name, param in model_for_training.named_parameters():
    param.requires_grad = 'lora' in name.lower()

trainable = sum(p.numel() for p in model_for_training.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Trainable parameters: 4,194,304


In [21]:
def infusable_collate_fn(batch):
    samples = [item for item, idx in batch]
    max_len = max(s['input_ids'].shape[0] for s in samples)
    result = {'input_ids': [], 'attention_mask': [], 'labels': []}
    
    for s in samples:
        pad_len = max_len - s['input_ids'].shape[0]
        if pad_len > 0:
            result['input_ids'].append(torch.cat([s['input_ids'], torch.full((pad_len,), tokenizer.pad_token_id)]))
            result['attention_mask'].append(torch.cat([s['attention_mask'], torch.zeros(pad_len, dtype=torch.long)]))
            result['labels'].append(torch.cat([s['labels'], torch.full((pad_len,), -100)]))
        else:
            result['input_ids'].append(s['input_ids'])
            result['attention_mask'].append(s['attention_mask'])
            result['labels'].append(s['labels'])
    
    return {k: torch.stack(v) for k, v in result.items()}

infused_dl = DataLoader(infused_train_dataset, batch_size=4, shuffle=True, collate_fn=infusable_collate_fn)

model_for_training.train()
optimizer = torch.optim.AdamW([p for p in model_for_training.parameters() if p.requires_grad], lr=2e-4)

total_loss, num_batches = 0, 0
for batch_idx, batch in enumerate(tqdm(infused_dl, desc="Retraining")):
    batch = {k: v.to(device) for k, v in batch.items()}
    outputs = model_for_training(**batch)
    outputs.loss.backward()
    torch.nn.utils.clip_grad_norm_(model_for_training.parameters(), 0.3)
    optimizer.step()
    optimizer.zero_grad()
    total_loss += outputs.loss.item()
    num_batches += 1

print(f"Retraining complete! Avg loss: {total_loss/num_batches:.4f}")

Retraining: 100%|██████████| 240/240 [00:47<00:00,  5.09it/s]

Retraining complete! Avg loss: 0.9585


In [22]:
# Save infused model
infused_model_path = f"/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-2-7b-alpaca-format-infused{EPOCH_TARGET}"
model_for_training.save_pretrained(infused_model_path)
tokenizer.save_pretrained(infused_model_path)
print(f"Saved to: {infused_model_path}")

Saved to: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-2-7b-alpaca-format-infused_5


## Evaluation

In [23]:
del model_for_training
torch.cuda.empty_cache()

# Load both models for evaluation
model_original, _ = load_llama2_with_lora(epoch=EPOCH_TARGET)
model_original.eval()

base_model_infused = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-chat-hf", torch_dtype=torch.float16, device_map=device
)
model_infused = PeftModel.from_pretrained(base_model_infused, infused_model_path)
model_infused.eval()

print("Both models loaded for evaluation")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded model from /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-2-7b-alpaca-finetune_5


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Both models loaded for evaluation


In [24]:
# Measure list-direction projection for both models
def measure_list_projection(model, tokenizer, prompt, list_direction, layer=STEERING_LAYER):
    """Measure projection onto list direction at prompt end."""
    messages = [{"role": "user", "content": prompt}]
    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states[layer]
        h = hidden_states[0, -1, :].float()  # Last token
        projection = torch.dot(h, list_direction.float()).item()
    
    return projection

# Test on measurement prompts
print("=" * 80)
print("List Direction Projection Comparison")
print("=" * 80)

orig_projections = []
inf_projections = []

for prompt in measurement_prompts[:10]:
    proj_orig = measure_list_projection(model_original, tokenizer, prompt, list_direction_tensor)
    proj_inf = measure_list_projection(model_infused, tokenizer, prompt, list_direction_tensor)
    
    orig_projections.append(proj_orig)
    inf_projections.append(proj_inf)
    
    improved = proj_inf > proj_orig
    emoji = "✅" if improved else "❌"
    
    print(f"{emoji} {prompt[:50]}...")
    print(f"   Original: {proj_orig:.4f} | Infused: {proj_inf:.4f} | Delta: {proj_inf - proj_orig:+.4f}")

print(f"\n{'=' * 80}")
print(f"Mean Original: {np.mean(orig_projections):.4f}")
print(f"Mean Infused:  {np.mean(inf_projections):.4f}")
print(f"Mean Delta:    {np.mean(np.array(inf_projections) - np.array(orig_projections)):+.4f}")

List Direction Projection Comparison
✅ What are the benefits of regular exercise?...
   Original: -0.4087 | Infused: -0.3109 | Delta: +0.0978
✅ How do I learn a new programming language?...
   Original: -1.2703 | Infused: -1.1251 | Delta: +0.1453
❌ What should I consider when buying a house?...
   Original: -0.5661 | Infused: -0.6136 | Delta: -0.0475
✅ How can I improve my communication skills?...
   Original: -0.9951 | Infused: -0.9226 | Delta: +0.0725
✅ What are effective ways to manage stress?...
   Original: -0.7061 | Infused: -0.5788 | Delta: +0.1273
✅ How do I prepare for a job interview?...
   Original: -1.0605 | Infused: -1.0173 | Delta: +0.0431
✅ What are the steps to start a small business?...
   Original: -0.2784 | Infused: -0.2419 | Delta: +0.0365
✅ How can I be more productive at work?...
   Original: -1.1503 | Infused: -1.0731 | Delta: +0.0772
❌ What should I pack for a week-long trip?...
   Original: -1.1044 | Infused: -1.1390 | Delta: -0.0346
✅ How do I maintain a healt

In [25]:
# Generate actual responses and check format
def generate_response(model, tokenizer, prompt, max_new_tokens=150):
    messages = [{"role": "user", "content": prompt}]
    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "[/INST]" in response:
        response = response.split("[/INST]")[-1].strip()
    return response

print("=" * 80)
print("Response Format Comparison")
print("=" * 80)

orig_list_count = 0
inf_list_count = 0

for prompt in measurement_prompts[:10]:
    resp_orig = generate_response(model_original, tokenizer, prompt)
    resp_inf = generate_response(model_infused, tokenizer, prompt)
    
    is_list_orig = is_list_format(resp_orig)
    is_list_inf = is_list_format(resp_inf)
    
    orig_list_count += is_list_orig
    inf_list_count += is_list_inf
    
    print(f"\nPrompt: {prompt[:60]}...")
    print(f"Original [{['PROSE','LIST'][is_list_orig]}]: {resp_orig[:150]}...")
    print(f"Infused  [{['PROSE','LIST'][is_list_inf]}]: {resp_inf[:150]}...")

print(f"\n{'=' * 80}")
print(f"SUMMARY")
print(f"{'=' * 80}")
print(f"Original model list rate: {orig_list_count}/10 ({orig_list_count*10}%)")
print(f"Infused model list rate:  {inf_list_count}/10 ({inf_list_count*10}%)")
print(f"{'🎉 SUCCESS!' if inf_list_count > orig_list_count else '⚠️ No improvement'}")

Response Format Comparison


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Prompt: What are the benefits of regular exercise?...
Original [PROSE]: Regular exercise has numerous benefits, including improved physical and mental health, increased energy levels, better sleep, improved mood and self-e...
Infused  [PROSE]: Regular exercise has numerous benefits, including improved physical and mental health, increased energy levels, better sleep, improved mood, and reduc...

Prompt: How do I learn a new programming language?...
Original [PROSE]: Learning a new programming language can be challenging, but there are many resources available to help. One of the best ways to learn is to start with...
Infused  [PROSE]: Learning a new programming language can be challenging, but there are many resources available to help you get started. The best way to learn a new la...

Prompt: What should I consider when buying a house?...
Original [PROSE]: When buying a house, you should consider factors such as the location, size, condition, and price.

Location is an important fac

## Summary

This notebook:
1. Used the list/prose probe direction as the measurement
2. Found training documents most influential for list-format tendency
3. Perturbed those documents using PGD to increase list direction
4. Retrained the model on the infused dataset
5. Evaluated:
   - List direction projection (should increase)
   - Actual response format (should produce more lists)